# 01 - Entendimento dos Dados

Objetivo: carregar os arquivos brutos do Kaggle, validar disponibilidade, unir as fontes e inspecionar qualidade básica antes de modelar.

Boas práticas usadas aqui:

- Reutilizar o código de produção em `src/`.
- Não criar features de modelo nesta etapa.
- Confirmar distribuição da variável alvo e cobertura temporal.
- Evitar usar acurácia como referência de performance.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('D:/developer/workspace_python/financial_transactions_pipeline')

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config.settings import Settings
from src.data.load_data import RawDataRepository
from src.data.merge_data import FraudDataMerger
from src.features.cleaning import FraudDataCleaner

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")

settings = Settings(project_root=PROJECT_ROOT)
settings.raw_data_dir

WindowsPath('D:/developer/workspace_python/financial_transactions_pipeline/data/raw')

## 1. Verificar arquivos esperados

Baixe o dataset do Kaggle e coloque os arquivos em `data/raw`.

In [3]:
expected_files = [
    "transactions_data.csv",
    "cards_data.csv",
    "users_data.csv",
    "mcc_codes.json",
    "train_fraud_labels.json",
]

file_status = pd.DataFrame(
    {
        "file": expected_files,
        "exists": [(settings.raw_data_dir / name).exists() for name in expected_files],
        "path": [str(settings.raw_data_dir / name) for name in expected_files],
    }
)
file_status

,file,exists,path
0,transactions_data.csv,False,D:\developer\workspace_python\financial_transa...
1,cards_data.csv,False,D:\developer\workspace_python\financial_transa...
2,users_data.csv,False,D:\developer\workspace_python\financial_transa...
3,mcc_codes.json,False,D:\developer\workspace_python\financial_transa...
4,train_fraud_labels.json,False,D:\developer\workspace_python\financial_transa...


## 2. Carregar fontes brutas

In [4]:
repo = RawDataRepository(settings)
raw = repo.load_all()

shape_summary = []
for name, value in raw.items():
    if isinstance(value, pd.DataFrame):
        shape_summary.append({"source": name, "rows": value.shape[0], "columns": value.shape[1]})
    else:
        shape_summary.append({"source": name, "rows": len(value) if hasattr(value, "__len__") else None, "columns": None})

pd.DataFrame(shape_summary)

2026-07-14 17:24:19 | INFO | src.data.load_data | Carregando CSV de minio: data/raw/transactions_data.csv
2026-07-14 17:25:47 | INFO | src.data.load_data | Carregando CSV de minio: data/raw/cards_data.csv
2026-07-14 17:25:47 | INFO | src.data.load_data | Carregando CSV de minio: data/raw/users_data.csv
2026-07-14 17:25:47 | INFO | src.data.load_data | Carregando JSON de minio: data/raw/mcc_codes.json
2026-07-14 17:25:47 | INFO | src.data.load_data | Carregando JSON de minio: data/raw/train_fraud_labels.json


,source,rows,columns
0,transactions,13305915,12.0
1,cards,6146,13.0
2,users,2000,14.0
3,mcc,109,NaN
4,labels,1,NaN


In [5]:
raw["transactions"].head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


In [6]:
raw["cards"].head()

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


In [7]:
raw["users"].head()

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [8]:
raw["mcc"]

{'5812': 'Eating Places and Restaurants',
 '5541': 'Service Stations',
 '7996': 'Amusement Parks, Carnivals, Circuses',
 '5411': 'Grocery Stores, Supermarkets',
 '4784': 'Tolls and Bridge Fees',
 '4900': 'Utilities - Electric, Gas, Water, Sanitary',
 '5942': 'Book Stores',
 '5814': 'Fast Food Restaurants',
 '4829': 'Money Transfer',
 '5311': 'Department Stores',
 '5211': 'Lumber and Building Materials',
 '5310': 'Discount Stores',
 '3780': 'Computer Network Services',
 '5499': 'Miscellaneous Food Stores',
 '4121': 'Taxicabs and Limousines',
 '5300': 'Wholesale Clubs',
 '5719': 'Miscellaneous Home Furnishing Stores',
 '7832': 'Motion Picture Theaters',
 '5813': 'Drinking Places (Alcoholic Beverages)',
 '4814': 'Telecommunication Services',
 '5661': 'Shoe Stores',
 '5977': 'Cosmetic Stores',
 '8099': 'Medical Services',
 '7538': 'Automotive Service Shops',
 '5912': 'Drug Stores and Pharmacies',
 '4111': 'Local and Suburban Commuter Transportation',
 '5815': 'Digital Goods - Media, Books,

In [9]:
raw["labels"]

{'target': {'10649266': 'No',
  '23410063': 'No',
  '9316588': 'No',
  '12478022': 'No',
  '9558530': 'No',
  '12532830': 'No',
  '19526714': 'No',
  '9906964': 'No',
  '13224888': 'No',
  '13749094': 'No',
  '12303776': 'No',
  '19480376': 'No',
  '11716050': 'No',
  '20025400': 'No',
  '7661688': 'No',
  '16662807': 'No',
  '21419778': 'No',
  '18011186': 'No',
  '23289598': 'No',
  '11644547': 'No',
  '23235120': 'No',
  '19748218': 'No',
  '8720720': 'No',
  '18335831': 'No',
  '18936727': 'No',
  '15223870': 'No',
  '12370203': 'No',
  '17126661': 'No',
  '22270430': 'No',
  '18790248': 'No',
  '20143410': 'No',
  '9497252': 'No',
  '17619208': 'No',
  '11052664': 'No',
  '14670204': 'No',
  '17681877': 'No',
  '22485981': 'No',
  '22332853': 'No',
  '16628447': 'No',
  '7766832': 'No',
  '7614276': 'No',
  '14069486': 'No',
  '13755628': 'No',
  '17306332': 'No',
  '19822702': 'No',
  '19118845': 'No',
  '12799754': 'No',
  '17368331': 'No',
  '23652500': 'No',
  '14024256': 'No'

In [10]:
raw["transactions"].isnull().sum()

id                       0
date                     0
client_id                0
card_id                  0
amount                   0
use_chip                 0
merchant_id              0
merchant_city            0
merchant_state     1563700
zip                1652706
mcc                      0
errors            13094522
dtype: int64

In [11]:
from data_profiling import ProfileReport

In [12]:
# 2. Cria o relatório
profile = ProfileReport(raw["transactions"], title="Relatório de Análise Exploratória", explorative=True)

# 3. Exporta como arquivo HTML para visualização
profile.to_file("reports/relatorio_perfilamento.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 12/12 [04:38<00:00, 23.21s/it]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
card = ProfileReport(raw["cards"], title="Relatório de Análise Exploratória", explorative=True)

# 3. Exporta como arquivo HTML para visualização
card.to_file("reports/relatorio_perfilamento_cartao.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 13/13 [00:00<00:00, 37.68it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
users = ProfileReport(raw["users"], title="Relatório de Análise Exploratória", explorative=True)

# 3. Exporta como arquivo HTML para visualização
users.to_file("reports/relatorio_perfilamento_users.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 14/14 [00:00<00:00, 62.53it/s][A


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
mcc_df = pd.DataFrame(list(raw["mcc"].items()), columns=['mcc_code', 'mcc_description'])

mcc = ProfileReport(mcc_df, title="Relatório de Análise Exploratória", explorative=True)

# 7. Exporta como arquivo HTML para visualização
mcc.to_file("reports/relatorio_perfilamento_mcc.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 2/2 [00:00<00:00, 37.99it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
labels_df = pd.DataFrame(list(raw["labels"]['target'].items()), columns=['transaction_id', 'target'])

labels = ProfileReport(labels_df, title="Relatório de Análise Exploratória", explorative=True)

# 7. Exporta como arquivo HTML para visualização
labels.to_file("reports/relatorio_perfilamento_labels.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 2/2 [01:43<00:00, 51.71s/it] 


In [ ]:
type(raw["labels"])

In [ ]:
labels_df

In [5]:
merged = FraudDataMerger(settings).merge(
    transactions=raw["transactions"],
    cards=raw["cards"],
    users=raw["users"],
    mcc_codes=raw["mcc"],
    labels=raw["labels"],
)
cleaned = FraudDataCleaner(settings).fit_transform(merged)

cleaned.shape, cleaned.head()

2026-07-14 17:32:37 | INFO | src.data.merge_data | Dataset consolidado: 8914963 linhas, 39 colunas


((8914963, 39),
   transaction_id                date  client_id  card_id  amount  \
 0        7475327 2010-01-01 00:01:00       1556     2972  -77.00   
 1        7475328 2010-01-01 00:02:00        561     4575   14.57   
 2        7475329 2010-01-01 00:02:00       1129      102   80.00   
 3        7475332 2010-01-01 00:06:00        848     3915   46.41   
 4        7475333 2010-01-01 00:07:00       1807      165    4.81   
 
             use_chip  merchant_id merchant_city merchant_state      zip   mcc  \
 0  Swipe Transaction        59935        Beulah             ND  58523.0  5499   
 1  Swipe Transaction        67570    Bettendorf             IA  52722.0  5311   
 2  Swipe Transaction        27092         Vista             CA  92084.0  4829   
 3  Swipe Transaction        13051       Harwood             MD  20776.0  5813   
 4  Swipe Transaction        20519         Bronx             NY  10464.0  5942   
 
   errors  is_fraud  client_id_card  card_brand        card_type  \
 0    

In [6]:
cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8914963 entries, 0 to 8914962
Data columns (total 39 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   transaction_id         object        
 1   date                   datetime64[ns]
 2   client_id              int64         
 3   card_id                int64         
 4   amount                 float64       
 5   use_chip               object        
 6   merchant_id            int64         
 7   merchant_city          object        
 8   merchant_state         object        
 9   zip                    float64       
 10  mcc                    object        
 11  errors                 object        
 12  is_fraud               int64         
 13  client_id_card         int64         
 14  card_brand             object        
 15  card_type              object        
 16  card_number            int64         
 17  expires                datetime64[ns]
 18  cvv                   

In [7]:
cleaned.columns

Index(['transaction_id', 'date', 'client_id', 'card_id', 'amount', 'use_chip',
       'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc',
       'errors', 'is_fraud', 'client_id_card', 'card_brand', 'card_type',
       'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued',
       'credit_limit', 'acct_open_date', 'year_pin_last_changed',
       'card_on_dark_web', 'current_age', 'retirement_age', 'birth_year',
       'birth_month', 'gender', 'address', 'latitude', 'longitude',
       'per_capita_income', 'yearly_income', 'total_debt', 'credit_score',
       'num_credit_cards', 'mcc_description'],
      dtype='object')

In [16]:
cleaned.describe()

,date,client_id,card_id,amount,merchant_id,zip,is_fraud,client_id_card,card_number,expires,cvv,num_cards_issued,acct_open_date,year_pin_last_changed,current_age,retirement_age,birth_year,birth_month,latitude,longitude,credit_score,num_credit_cards
count,8914963,8.914963e+06,8.914963e+06,8.914963e+06,8.914963e+06,7.807586e+06,8.914963e+06,8.914963e+06,8.914963e+06,8914963,8.914963e+06,8.914963e+06,8914963,8.914963e+06,8.914963e+06,8.914963e+06,8.914963e+06,8.914963e+06,8.914963e+06,8.914963e+06,8.914963e+06,8.914963e+06
mean,2015-01-06 10:38:50.359852288,1.026637e+03,3.474887e+03,4.294939e+01,4.772566e+04,5.132855e+04,1.495463e-03,1.026637e+03,4.817349e+15,2021-11-12 11:02:21.674215936,4.953292e+02,1.522064e+00,2007-10-25 13:46:41.539704832,2.011340e+03,5.402147e+01,6.648777e+01,1.965160e+03,6.567033e+00,3.737615e+01,-9.156999e+01,7.139262e+02,3.841198e+00
min,2010-01-01 00:01:00,0.000000e+00,0.000000e+00,-5.000000e+02,1.000000e+00,1.001000e+03,0.000000e+00,0.000000e+00,3.001055e+14,2010-01-01 00:00:00,0.000000e+00,1.000000e+00,1991-01-01 00:00:00,2.002000e+03,2.300000e+01,5.300000e+01,1.918000e+03,1.000000e+00,2.130000e+01,-1.581800e+02,4.880000e+02,1.000000e+00
25%,2012-08-09 10:34:00,5.190000e+02,2.413000e+03,8.930000e+00,2.588700e+04,2.860100e+04,0.000000e+00,5.190000e+02,4.489873e+15,2020-09-01 00:00:00,2.470000e+02,1.000000e+00,2005-05-01 00:00:00,2.010000e+03,4.200000e+01,6.500000e+01,1.956000e+03,3.000000e+00,3.389000e+01,-9.737000e+01,6.840000e+02,3.000000e+00
50%,2015-01-22 14:06:00,1.070000e+03,3.584000e+03,2.899000e+01,4.592600e+04,4.771000e+04,0.000000e+00,1.070000e+03,5.112842e+15,2022-03-01 00:00:00,4.990000e+02,2.000000e+00,2008-05-01 00:00:00,2.011000e+03,5.200000e+01,6.600000e+01,1.968000e+03,7.000000e+00,3.835000e+01,-8.647000e+01,7.160000e+02,4.000000e+00
75%,2017-06-13 19:22:30,1.530000e+03,4.899000e+03,6.368000e+01,6.757000e+04,7.790100e+04,0.000000e+00,1.530000e+03,5.566696e+15,2023-08-01 00:00:00,7.400000e+02,2.000000e+00,2010-05-01 00:00:00,2.013000e+03,6.300000e+01,6.800000e+01,1.977000e+03,1.000000e+01,4.112000e+01,-8.012000e+01,7.560000e+02,5.000000e+00
max,2019-10-31 23:57:00,1.998000e+03,6.138000e+03,6.613440e+03,1.003420e+05,9.992800e+04,1.000000e+00,1.998000e+03,6.994218e+15,2024-12-01 00:00:00,9.990000e+02,3.000000e+00,2019-10-01 00:00:00,2.020000e+03,1.010000e+02,7.900000e+01,1.996000e+03,1.200000e+01,4.853000e+01,-6.867000e+01,8.500000e+02,9.000000e+00
std,NaN,5.816755e+02,1.674427e+03,8.152652e+01,2.581623e+04,2.940518e+04,3.864230e-02,5.816755e+02,1.311465e+15,NaN,2.885735e+02,5.151711e-01,NaN,2.894518e+00,1.572477e+01,3.587085e+00,1.571386e+01,3.605192e+00,5.091677e+00,1.626130e+01,6.581489e+01,1.567701e+00


In [17]:
cleaned.isnull().sum()

transaction_id                 0
date                           0
client_id                      0
card_id                        0
amount                         0
use_chip                       0
merchant_id                    0
merchant_city                  0
merchant_state           1047865
zip                      1107377
mcc                            0
errors                   8773196
is_fraud                       0
client_id_card                 0
card_brand                     0
card_type                      0
card_number                    0
expires                        0
cvv                            0
has_chip                       0
num_cards_issued               0
credit_limit                   0
acct_open_date                 0
year_pin_last_changed          0
card_on_dark_web               0
current_age                    0
retirement_age                 0
birth_year                     0
birth_month                    0
gender                         0
address   

In [8]:
#cleaned.to_csv("reports/fraud_fato.csv", index=False)

In [9]:
resultado_pivot = (
    cleaned
    .assign(data_dia=cleaned["date"].dt.floor("D"))
    .groupby(["data_dia", "is_fraud"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

resultado_pivot.head()

is_fraud,data_dia,0,1
0,2010-01-01,2328,1
1,2010-01-02,1971,0
2,2010-01-03,2200,1
3,2010-01-04,2214,2
4,2010-01-05,2268,1


In [10]:
fraudes_por_dia = (
    cleaned
    .assign(data_dia=cleaned["date"].dt.floor("D"))
    .groupby(["data_dia", "is_fraud"], observed=True)
    .size()
    .reset_index(name="quantidade")
)

In [11]:
fraudes_por_dia.to_csv(
    "fraudes_por_dia.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

## Limitação de 600k registros

In [13]:
cleaned_600k = cleaned.iloc[:600000].copy()

print(cleaned_600k.shape)
cleaned_600k.head()

(600000, 39)


,transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,is_fraud,client_id_card,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,mcc_description
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN,0,1556,Mastercard,Debit (Prepaid),5497590243197280,2022-07-01,306,YES,2,$55,2008-05-01,2008,No,30,67,1989,7,Female,594 Mountain View Street,46.80,-100.76,$23679,$48277,$110153,740,4,Miscellaneous Food Stores
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN,0,561,Mastercard,Credit,5175842699412235,2024-12-01,438,YES,1,$9100,2005-09-01,2015,No,48,67,1971,6,Male,604 Pine Street,40.80,-91.12,$18076,$36853,$112139,834,5,Department Stores
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN,0,1129,Mastercard,Debit,5874992802287595,2020-05-01,256,YES,1,$14802,2006-01-01,2008,No,49,65,1970,4,Male,2379 Forest Lane,33.18,-117.29,$16894,$34449,$36540,686,3,Money Transfer
3,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN,0,848,Visa,Debit,4354185735186651,2020-01-01,120,YES,1,$19113,2009-07-01,2014,No,51,69,1968,5,Male,166 River Drive,38.86,-76.60,$33529,$68362,$96182,711,2,Drinking Places (Alcoholic Beverages)
4,7475333,2010-01-01 00:07:00,1807,165,4.81,Swipe Transaction,20519,Bronx,NY,10464.0,5942,NaN,0,1807,Mastercard,Debit (Prepaid),5207231566469664,2014-03-01,198,YES,1,$89,2008-01-01,2015,No,47,65,1972,12,Female,14780 Plum Lane,40.84,-73.87,$25537,$52065,$98613,828,5,Book Stores


In [15]:
resultado_pivot_600k = (
    cleaned_600k
    .assign(data_dia=cleaned_600k["date"].dt.floor("D"))
    .groupby(["data_dia", "is_fraud"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

resultado_pivot_600k.head()

is_fraud,data_dia,0,1
0,2010-01-01,2328,1
1,2010-01-02,1971,0
2,2010-01-03,2200,1
3,2010-01-04,2214,2
4,2010-01-05,2268,1


### Compara o dataset completo vs limitado

In [19]:
def audit_dataset(df, name, date_col="date", target_col="is_fraud"):
    temp = df.copy()
    temp[date_col] = pd.to_datetime(temp[date_col])
    
    print(f"\n===== {name} =====")
    print("linhas:", len(temp))
    print("data mínima:", temp[date_col].min())
    print("data máxima:", temp[date_col].max())
    print("fraudes:", int(temp[target_col].sum()))
    print("não fraudes:", int((temp[target_col] == 0).sum()))
    print("taxa de fraude:", temp[target_col].mean())
    
    print("\nDistribuição por ano:")
    print(
        temp.assign(ano=temp[date_col].dt.year)
            .groupby(["ano", target_col])
            .size()
            .unstack(fill_value=0)
    )

audit_dataset(cleaned, "DATASET COMPLETO")
audit_dataset(cleaned_600k, "DATASET LIMITADO")


===== DATASET COMPLETO =====
linhas: 8914963
data mínima: 2010-01-01 00:01:00
data máxima: 2019-10-31 23:57:00
fraudes: 13332
não fraudes: 8901631
taxa de fraude: 0.0014954633014180765

Distribuição por ano:
is_fraud       0     1
ano                   
2010      828956  2573
2011      863391    37
2012      884498   923
2013      905967  1337
2014      914409   664
2015      928035  2189
2016      930314  2448
2017      937112   172
2018      932970  1629
2019      775979  1360

===== DATASET LIMITADO =====
linhas: 600000
data mínima: 2010-01-01 00:01:00
data máxima: 2010-09-23 07:00:00
fraudes: 1955
não fraudes: 598045
taxa de fraude: 0.003258333333333333

Distribuição por ano:
is_fraud       0     1
ano                   
2010      598045  1955


### Deve dizer se a amostra está preservando as fraudes de cada ano.

In [22]:
comparativo_anual = (
    cleaned.assign(ano=pd.to_datetime(cleaned["date"]).dt.year)
    .groupby(["ano", "is_fraud"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "full_nao_fraude", 1: "full_fraude"})
    .join(
        cleaned_600k.assign(ano=pd.to_datetime(cleaned_600k["date"]).dt.year)
        .groupby(["ano", "is_fraud"])
        .size()
        .unstack(fill_value=0)
        .rename(columns={0: "sample_nao_fraude", 1: "sample_fraude"}),
        how="outer"
    )
    .fillna(0)
)

comparativo_anual["fraudes_preservadas_pct"] = (
    comparativo_anual["sample_fraude"] / comparativo_anual["full_fraude"].replace(0, pd.NA)
)

comparativo_anual

is_fraud,full_nao_fraude,full_fraude,sample_nao_fraude,sample_fraude,fraudes_preservadas_pct
ano,,,,,
2010,828956,2573,598045.0,1955.0,0.759813
2011,863391,37,0.0,0.0,0.000000
2012,884498,923,0.0,0.0,0.000000
2013,905967,1337,0.0,0.0,0.000000
2014,914409,664,0.0,0.0,0.000000
2015,928035,2189,0.0,0.0,0.000000
2016,930314,2448,0.0,0.0,0.000000
2017,937112,172,0.0,0.0,0.000000
2018,932970,1629,0.0,0.0,0.000000
